# មេរៀន ០១ - ការណែនាំអំពីភ្នាក់ងារបញ្ញាសិប្បនិម្មិត

ស្វាគមន៍មកកាន់មេរៀនដំបូងក្នុងវគ្គសិក្សា **ភ្នាក់ងារបញ្ញាសិប្បនិម្មិតសម្រាប់អ្នកចាប់ផ្តើម**!

ភ្នាក់ងារបញ្ញាសិប្បនិម្មិត (**AI agent**) គឺជាកម្មវិធីមួយដែលប្រើម៉ូឌែលភាសាធំ (LLM) ជាសៀវភៅលំហាត់លទ្ធផល និងអាចចាត់វិធីសាស្ត្រដ៏ទាក់ទងក្នុងពិភពជាក់ស្តែង — ចូលប្រើ API, សាកសួរទិន្នន័យឃ្លាំង, ឬរត់កូដ — ដើម្បីបំពេញគោលបំណងមួយជំនួសអ្នកប្រើប្រាស់។

ក្នុងសៀវភៅកំណត់ត្រានេះ លោកអ្នកនឹងសាងសង់ភ្នាក់ងារដំបូងរបស់អ្នក: **ភ្នាក់ងារ​ទេសចរណ៍** ដែលផ្តល់អនុសាសន៍គោលដៅចុះសម្រាក។ តាមដានការ អ្នកនឹងរៀនពីរបៀប:

1. តភ្ជាប់ទៅកាន់សេវាកម្ម Microsoft Foundry Agent ដោយប្រើ **Microsoft Agent Framework**។
2. ផ្តល់ឧបករណ៍មួយសម្រាប់ភ្នាក់ងារ — មុខងារ Python បូមាត់មួយដែលវាអាចហៅបាន។
3. បើករត់ភ្នាក់ងារនិងពិនិត្យមើលចម្លើយរបស់វា។
4. ស្ទ្រីមចម្លើយភ្នាក់ងារtoken បន្ទាប់ token។


## ការតំឡើង

មុនពេលរត់ notebook នេះ សូមប្រាកដថាអ្នកមាន:

1. **គម្រោង Microsoft Foundry** មានម៉ូដែលសន្ទនាត្រូវបានដាក់ឲ្យប្រើ (ឧ. `gpt-4.1-mini`)។
2. **បានចូលប្រើជាមួយ Azure CLI** — រត់ `az login` នៅក្នុង terminal របស់អ្នក។
3. **កំណត់អថេរបរិស្ថានដែលត្រូវការ៖**
   - `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចចេញគម្រោង Microsoft Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះម៉ូដែលដែលបានដាក់ឲ្យប្រើរបស់អ្នក។

ក្រុមកូដខាងក្រោមដំឡើងកញ្ចប់ Python ដែលអ្នកត្រូវការជាដើម។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ការបង្កើតអេហ្សិនត៍ដំបូងរបស់អ្នក

អេហ្សិនត៍ត្រូវការពីរពីរោះ:

- **ការណែនាំ** ដែលប្រាប់វា *តើវា​ជាអ្នកណា* និង *របៀបដែលត្រូវឆ្លូតឆ្លង* (សេចក្ដីណែនាំប្រព័ន្ធ)។
- **ឧបករណ៍** — មុខងារបាយ Python ដែលត្រូវ​បានតុបតែងជាមួយ `@tool` ដែលអេហ្សិនត៍អាចហៅដើម្បីយកព័ត៌មានឬបំពេញសកម្មភាព។

ខាងក្រោមយើងកំណត់ឧបករណ៍សាមញ្ញមួយដែលបង្វិលត្រឡប់បញ្ជីទីកន្លែង​សម្រាកខ្លួនពេញនិយម។ អេហ្សិនត៍​នឹងប្រើឧបករណ៍នេះនៅពេលមនុស្សប្រើស្នើសុំការផ្ដល់អនុសាសន៍ធ្វើដំណើរ។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ការឆ្លើយតបបញ្ចាំងបណ្ដាញ

សម្រាប់បទពិសោធន៍ដែលអាចចូលរួមបានច្រើនជាងនេះ អ្នកអាច **បញ្ចាំងបញ្ចេញ** ការឆ្លើយតបរបស់ភ្នាក់ងារ។ ជំនួសការរងចាំចម្លើយពេញលេញ ភ្នាក់ងារបញ្ចេញខ្នាតអក្សរដូចជាតែត្រូវបានបង្កើត។ វាមានប្រយោជន៍យ៉ាងខ្លាំងនៅក្នុងចំណុចជជែកដែលអ្នកចង់បង្ហាញលទ្ធផលក្នុងពេលជាក់លាក់។


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## រួមសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនអំពីរបៀប:

- **បង្កើតអ្នកផ្គត់ផ្គង់** ដែលភ្ជាប់ទៅកាន់សេវាកម្ម Microsoft Foundry Agent តាមរយៈ `FoundryChatClient`។
- **កំណត់ឧបករណ៍** ដោយប្រើ `@tool` decorator ដើម្បីឲ្យភ្នាក់ងារអាចហៅមុខងារ Python របស់អ្នក។
- **រត់ភ្នាក់ងារ** ជាមួយសារអ្នកប្រើ ហើយបោះពុម្ពចេញនូវចម្លើយរបស់វា។
- **ផ្ទុកចរន្តចម្លើយ** សម្រាប់លទ្ធផលពេលវេលាពិត។

នៅក្នុងមេរៀនបន្ទាប់ យើងនឹងស្វែងយល់ជ្រាបជាន់ខ្ពស់ពីស្រទាប់ហេតុបច្ចេកទេស agentic និងរៀនពីរបៀបផ្តល់ឧបករណ៍មានសម្លេងខ្លាំង និងសមត្ថភាពគិតលំដាប់ច្រើនជំហានដល់ភ្នាក់ងារ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
